## Kalman filter for options

In [1]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.massive.client
%aimport quantdata.massive.processing
%aimport quantdata.massive.options.contracts

In [ ]:



from datetime import date
from cdxcore.jcpool import JCPool
from cdxcore.pretty import PrettyObject
from cdxcore.subdir import SubDir
from cdxcore.util import fmt_list
from cdxcore.uniquehash import named_unique_filename48_8
import polars as pl

from quantdata.biz_calendar import prev_business_day
from quantdata.massive.client import MassiveClient
from quantdata.pool import PoolConfig
from quantdata.massive.options.contracts import ContractsFilter, filter_contracts, REJECTED
from quantdata.massive.processing import TradeQuoteAlignmentConfig, align_trades_to_quotes, align_quotes_to_trades, filter_valid_quotes, filter_valid_trades

from cdxcore.version import version
from cdxcore.verbose import Context
from dataclasses import asdict, dataclass, field
from cdxcore.uniquehash import unique_hash8 
import numpy as np
import math
import warnings

from quantdata.ncde.implied.day import _ncde_implied_day_root as _cache
from IPython.display import display, Markdown, clear_output


@version("0.2", dependencies=[ContractsFilter])
@dataclass(frozen=True)
class KalmanConfig:
    contract_filter : ContractsFilter = ContractsFilter()
    max_num_options_per_expiry : int|None = 1000
    trade_match_lookback_ms: int = 250
    trade_match_lookfwd_ms: int = 250

    def __str__(self):
        return f"mxn={self.max_num_options_per_expiry},{self.contract_filter}"

@_cache(fmt=SubDir.POLARS_PARQUET).cache(
    "0.1",dependencies=[align_quotes_to_trades],
    uid=lambda ticker, d: f"{ticker}/{d}/{ticker}"
)
def ticker_trade_series( ticker : str, d : date, *, verbose : Context|None = Context.quiet) -> pl.DataFrame:
    """ Cached ticker series """
    verbose     = verbose if verbose is not None else Context.quiet
    with verbose.write_t(lambda : f"{ticker} {d}: Fetching time series for {ticker} {d}... ", end='') as tme:
        df = align_quotes_to_trades( ticker, d, within_session=True, drop_unmatched=False )
        verbose.write(lambda : f"done; found {len(df)} quotes; this took {tme}.", head=False)
    return df.sort("ts_ns")
    
@_cache(fmt=SubDir.POLARS_PARQUET).cache(
    "0.14",
    dependencies=[align_trades_to_quotes,ticker_trade_series],
    uid=lambda ticker, opt_ticker, d: f"{ticker}/{d}/{named_unique_filename48_8(opt_ticker)}"
)
def opt_quote_time_series( ticker : str, opt_ticker : str, d : date, *, verbose : Context|None = Context.quiet):
    """
    Option quote time series aligned with spot trades.
    """
    verbose     = verbose if verbose is not None else Context.quiet
    spot_trades = ticker_trade_series( ticker, d, verbose=verbose )

    """
    spot_trades schema:
        SCHEMA_ALIGN_QUOTES_TO_TRADES = {
            "ts_ns": np.dtype("datetime64[ns]"),
            "price": np.float64,
            "size": np.float64,
            "trade_code": np.uint32,
            "quote_ts_ns": np.dtype("datetime64[ns]"),
            "bid_price": np.float64,
            "bid_size": np.float64,
            "ask_price": np.float64,
            "ask_size": np.float64,
        }
    """

    with verbose.write_t(lambda : f"{ticker} {d}: Fetching time series for option {opt_ticker} {d}... ", end='') as tme:
        opt_quotes  = align_trades_to_quotes( opt_ticker, d, within_session=True, drop_unmatched=False ).sort("ts_ns")
        if len(opt_quotes) == 0:
            client     = MassiveClient()
            num_quotes = len( client.fetch_quotes(opt_ticker, d) )
            num_trades = len( client.fetch_trades(opt_ticker, d) )
            num_valid_quotes = len( filter_valid_quotes( opt_ticker, d))
            num_valid_trades = len( filter_valid_trades( opt_ticker, d))

            warnings.warn(f"{ticker} {d} option {opt_ticker}: No valid quotes were aligned with valid trades. Number of quotes: {num_quotes}, valid quotes: {num_valid_quotes}, number of trades: {num_trades}, valid trades: {num_valid_trades}.")
            return opt_quotes
        verbose.write(lambda : f"done; found {len(opt_quotes)} quotes; this took {tme}.", head=False)

    """
    opt_quotes schema:
        SCHEMA_ALIGN_TRADES_TO_QUOTES = {
            "ts_ns": np.dtype("datetime64[ns]"),
            "bid_price": np.float64,
            "bid_size": np.float64,
            "ask_price": np.float64,
            "ask_size": np.float64,
            "trade_ts_ns": np.dtype("datetime64[ns]"),
            "price": np.float64,
            "size": np.float64,
            "trade_code": np.uint32,
        }    
    """

    spot_cols   = set(spot_trades.columns)
    spot_trades = spot_trades.rename({k: "spot_" + k for k in spot_cols})

    result = opt_quotes.join_asof(
            spot_trades,
            left_on="ts_ns",
            right_on="spot_ts_ns",
            strategy="backward" )
    result      = result.with_columns( *[ pl.col("spot_" + c).forward_fill().backward_fill() for c in spot_cols ] )
    result      = result.filter(pl.col("bid_price").is_not_null())
    assert len(result) > 0, f"{ticker} {d} option {opt_ticker}: No aligned quotes found for option {opt_ticker} on {d} after joining {len(opt_quotes)} quites with {len(spot_trades)} spot trades...?"
    return result

def generate_day( ticker : str, d : date, 
                  kalman_config : KalmanConfig, *,
                  pool_config : PoolConfig = PoolConfig.off(),
                  verbose     : Context = Context.quiet
                  ):
    """
    Generate polars data frame per expiry for all eligible options.
    """

    verbose.write(lambda : f"{ticker} {d}: Starting data generation with config {kalman_config} and pool config {pool_config}")

    # contract selection
    # ------------------

    df = filter_contracts( ticker, d=d, exclude_half_days=True, drop_no_trades_today=True,
                           contracts_filter=kalman_config.contract_filter, pool_config=pool_config, verbose=verbose )
    df = df.filter(pl.col("reject_code") == REJECTED.USED)

    # reference data
    # --------------

    expiries  = df["exp_date"].unique().sort().to_list()
    prev_spot = float( df["spot_ref_prev_close"].unique()[0] )

    ref_data    = ( df.with_columns( pl.format("{} {} DTE {} {} {} @ {} ({}%)", pl.col("ticker"), pl.col("exp_date"), pl.col("days_to_exp"), pl.col("exercise_style"), pl.col("contract_type"), pl.col("strike_price").round(2), (100.0 * pl.col("strike_price")/prev_spot).round(2)).alias("opt_str") ).
                    select( pl.col("ticker").alias("opt_ticker"), 
                            pl.col("exp_date"), 
                            pl.col("nms5"),
                            pl.col("exercise_style")
                                .str.to_lowercase()
                                .str.starts_with("a")
                                .alias("is_american"),
                            pl.col("contract_type")
                                .str.to_lowercase()
                                .str.starts_with("c")
                                .alias("is_call"),                                   
                            ( pl.col("strike_price")/prev_spot ).alias("strike"),
                                pl.col("days_to_exp").alias("dte"), 
                                pl.lit(prev_spot).alias("prev_spot") ).with_columns( pl.lit(ticker).alias("ticker") )
            )
    ref_data      = { expiry: ref_data.filter(pl.col("exp_date") == expiry).drop("exp_date") for expiry in expiries }

    if kalman_config.max_num_options_per_expiry is not None:
        verbose.write(lambda : f"{ticker} {d}: Limiting to at most {kalman_config.max_num_options_per_expiry} options per expiry, selected by smallest absolute nms5:")
        for expiry, ref_df in ref_data.items():
            n_cnt = len(ref_df)
            ref_df   = ref_df.with_columns(abs_nms5=pl.col("nms5").abs()).sort("abs_nms5").head(kalman_config.max_num_options_per_expiry).drop("abs_nms5")
            min_nms5 = ref_df["nms5"].min()
            max_nms5 = ref_df["nms5"].max()
            verbose.report(1,lambda : f"{ticker} {d} expiry {expiry}: Selected {len(ref_df)}/{n_cnt} options, sorted by smallest absolute nms5. Resulting options range from nms5 {min_nms5:.3f} to {max_nms5:.3f}.")

    # time series
    # -----------

    opt_tickers    = set()
    opt_to_expiry  = {}
    for expiry, exp_ref_data in ref_data.items():
        exp_opt_tickers = set(exp_ref_data["opt_ticker"].unique().to_list())
        assert len(opt_tickers & exp_opt_tickers) == 0, f"{ticker} {d}: Mismatch in option tickers for expiry {expiry}: expected {opt_tickers^exp_opt_tickers} already seen...?"
        opt_tickers     |= exp_opt_tickers
        for opt_ticker in exp_opt_tickers:
            opt_to_expiry[opt_ticker] = expiry
        del exp_opt_tickers
    del expiry, exp_ref_data

    with verbose.write_t(lambda : f"{ticker} {d}: Computing enriched time series for {len(opt_tickers)} options for expiries {fmt_list(sorted(expiries))}:") as tme:
        def pull():
            pool          = JCPool( **asdict(pool_config) )
            is_single     = pool_config.num_workers in [0,1] and pool_config.threading
            results       = { e: {} for e in expiries }
            cnt           = 0
            tme           = verbose.timer()
            excluded      = set()

            for opt_ticker, _ in pool.parallel( { opt_ticker: pool.delayed( opt_quote_time_series )( ticker, opt_ticker, d, verbose=verbose(1) if is_single else None, cache_generate_only=True ) for opt_ticker in opt_tickers } ):
                cnt += 1
                r = opt_quote_time_series( ticker, opt_ticker, d, verbose=Context.quiet, override_cache_mode="cacheonly" )

                if len(r) == 0:
                    verbose.report(1,lambda : f"\r{ticker} {d}: Processing option {opt_ticker} {cnt}/{len(opt_tickers)}: this option had no valid quotes -- {math.ceil(float(cnt)/float(len(opt_tickers))*100)}% done; so far this took {tme}...", end='')
                    excluded |= { opt_ticker }
                else:
                    if cnt==1 or tme.interval_test(1.):
                        verbose.report(1,lambda : f"\r{ticker} {d}: Processing option {opt_ticker} {cnt}/{len(opt_tickers)} ... {math.ceil(float(cnt)/float(len(opt_tickers))*100)}% done; so far this took {tme}...", end='')

                    r = r.with_columns( pl.lit(opt_ticker).alias("opt_ticker") )
                    expiry = opt_to_expiry[opt_ticker]
                    results[expiry][opt_ticker] = r
            return results, excluded

        results, excluded = pull()

        assert set(results) == set(ref_data), f"{ticker} {d}: Mismatch in expiries between results and reference data: expected {sorted(expiries)}, got {sorted(results)}, expected {sorted(ref_data)}"
        assert set(ref_data) == set(expiries), f"{ticker} {d}: Mismatch in expiries between results and reference data: expected {sorted(expiries)}, got {sorted(results)}, expected {sorted(ref_data)}"

        # merge per-option results into per-expiry results
        for expiry, opt_dict in results.items():
            expiry_ref_data = ref_data.pop(expiry)

            opt_tickers     = set( expiry_ref_data["opt_ticker"].to_list() )
            opt_tick_d      = set( opt_dict )

            assert opt_tickers - opt_tick_d <= excluded, f"Internal error: {ticker} {d}: Mismatch in options for expiry {expiry}: expected {sorted(opt_tick_d)}, got {sorted(opt_tickers)} (outside {excluded})"

            opt_dfs = []
            for opt_ticker, df in opt_dict.items():
                opt_dfs.append( df )
            opt_dfs         = pl.concat(opt_dfs).sort("ts_ns")
            opt_tickers     = set( opt_dict )
            df_tickers      = set( opt_dfs["opt_ticker"].unique().to_list() )
            if opt_tickers != df_tickers:
                missing = opt_tickers - df_tickers
                extra   = df_tickers - opt_tickers
                assert opt_tickers == df_tickers, f"{ticker} {d}: Mismatch in tickers for expiry {expiry}: df_ticker is missing: {fmt_list(missing)}, df_ticker has extra: {fmt_list(extra)}"
            results[expiry] = dict(opt_data=opt_dfs, ref_data=expiry_ref_data)

        verbose.write(lambda : f"\r{ticker} {d}: Computed enriched time series for {len(df)} options; this took {tme}.")
    return results

clear_output()
d = prev_business_day(date(2025,1,23), exclude_half_days=True, count_start=True )
pool_config= PoolConfig(num_workers=0, threading=False) 
#df = opt_quote_time_series("SPY","O:SPY250623P00601000",d, verbose=Context.all)

results = generate_day("SPY", d, KalmanConfig(), pool_config=pool_config, verbose=Context.all)

for d, df_dict in results.items():
    display(Markdown(f"## Expiry {d}"))
    display(Markdown(f"#### Expiry {d} Opt Data"))
    display(df_dict["opt_data"])
    display(Markdown(f"#### Expiry {d} Ref Data"))
    display(df_dict["ref_data"])
"""
(-1) add_business_days: 2025-01-23 -1->-1 True True [datetime.date(2025, 1, 23), datetime.date(2025, 1, 22), datetime.date(2025, 1, 21), datetime.date(2025, 1, 17), datetime.date(2025, 1, 16)] -> 2025-01-16
(-2) add_business_days: 2025-01-23 -1->1 True True [datetime.date(2025, 1, 16), datetime.date(2025, 1, 17), datetime.date(2025, 1, 21), datetime.date(2025, 1, 22), datetime.date(2025, 1, 23)] -> 2025-01-17
"""

01:   SPY 2025-01-23: Processing option O:SPY250131C00596000 391/429 ... 92% done; so far this took 3s...00: SPY 2025-01-23: Starting data generation with config mxn=1000,max_dte=7,max_mns=2,min_opt_vol_k=0 and pool config workers 0p
00: SPY 2025-01-23: Limiting to at most 1000 options per expiry, selected by smallest absolute nms5:
01:   SPY 2025-01-23 expiry 2025-01-23: Selected 32/32 options, sorted by smallest absolute nms5. Resulting options range from nms5 -0.848 to 0.851.
01:   SPY 2025-01-23 expiry 2025-01-24: Selected 52/52 options, sorted by smallest absolute nms5. Resulting options range from nms5 -1.104 to 1.097.
01:   SPY 2025-01-23 expiry 2025-01-27: Selected 50/50 options, sorted by smallest absolute nms5. Resulting options range from nms5 -1.233 to 1.219.
01:   SPY 2025-01-23 expiry 2025-01-28: Selected 54/54 options, sorted by smallest absolute nms5. Resulting options range from nms5 -1.351 to 1.330.
01:   SPY 2025-01-23 expiry 2025-01-29: Selected 59/59 options, sorte

## Expiry 2025-01-23

#### Expiry 2025-01-23 Opt Data

ts_ns,bid_price,bid_size,ask_price,ask_size,trade_ts_ns,price,size,trade_code,spot_ts_ns,spot_price,spot_size,spot_trade_code,spot_quote_ts_ns,spot_bid_price,spot_bid_size,spot_ask_price,spot_ask_size,opt_ticker
"datetime[ns, America/New_York]",f64,f64,f64,f64,"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,f64,f64,str
2025-01-23 09:30:00.003641344 EST,0.01,126.0,0.02,126.0,2025-01-23 09:30:00.040 EST,0.02,11.0,64,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250123C00613000"""
2025-01-23 09:30:00.003656448 EST,0.02,119.0,0.03,126.0,2025-01-23 09:30:00.230 EST,0.03,1.0,2112,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250123C00611000"""
2025-01-23 09:30:00.005472512 EST,0.15,94.0,0.16,94.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250123P00601000"""
2025-01-23 09:30:00.005662976 EST,0.22,81.0,0.23,31.0,2025-01-23 09:30:00.005 EST,0.23,50.0,64,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250123P00602000"""
2025-01-23 09:30:00.007490560 EST,0.04,5.0,0.07,43.0,2025-01-23 09:30:00.008 EST,0.05,3.0,64,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250123C00610000"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-23 15:59:59.992190464 EST,6.89,1.0,8.71,104.0,null,null,null,null,2025-01-23 15:59:59.978344960 EST,609.71,100.0,0,2025-01-23 15:59:59.978341120 EST,609.71,16300.0,609.73,300.0,"""O:SPY250123C00602000"""
2025-01-23 15:59:59.992440832 EST,3.67,1.0,5.31,95.0,null,null,null,null,2025-01-23 15:59:59.978344960 EST,609.71,100.0,0,2025-01-23 15:59:59.978341120 EST,609.71,16300.0,609.73,300.0,"""O:SPY250123C00605000"""
2025-01-23 15:59:59.992781568 EST,9.33,10.0,11.89,94.0,null,null,null,null,2025-01-23 15:59:59.978344960 EST,609.71,100.0,0,2025-01-23 15:59:59.978341120 EST,609.71,16300.0,609.73,300.0,"""O:SPY250123C00599000"""


#### Expiry 2025-01-23 Ref Data

opt_ticker,nms5,is_american,is_call,strike,dte,prev_spot,ticker
str,f32,bool,bool,f32,i16,f64,str
"""O:SPY250123C00599000""",-0.847646,true,true,0.987732,0,606.440002,"""SPY"""
"""O:SPY250123C00600000""",-0.733105,true,true,0.989381,0,606.440002,"""SPY"""
"""O:SPY250123C00601000""",-0.618754,true,true,0.99103,0,606.440002,"""SPY"""
"""O:SPY250123C00602000""",-0.504593,true,true,0.992679,0,606.440002,"""SPY"""
"""O:SPY250123C00603000""",-0.390622,true,true,0.994328,0,606.440002,"""SPY"""
…,…,…,…,…,…,…,…
"""O:SPY250123P00610000""",0.401922,true,false,1.00587,0,606.440002,"""SPY"""
"""O:SPY250123P00611000""",0.5144,true,false,1.007519,0,606.440002,"""SPY"""
"""O:SPY250123P00612000""",0.626694,true,false,1.009168,0,606.440002,"""SPY"""


... (32 outputs left out)

ts_ns,bid_price,bid_size,ask_price,ask_size,trade_ts_ns,price,size,trade_code,spot_ts_ns,spot_price,spot_size,spot_trade_code,spot_quote_ts_ns,spot_bid_price,spot_bid_size,spot_ask_price,spot_ask_size,opt_ticker
"datetime[ns, America/New_York]",f64,f64,f64,f64,"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,u32,"datetime[ns, America/New_York]",f64,f64,f64,f64,str
2025-01-23 09:30:00.004868608 EST,0.47,23.0,0.53,13.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250203C00620000"""
2025-01-23 09:30:00.009810176 EST,0.5,15.0,0.52,88.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250203C00620000"""
2025-01-23 09:30:00.009990656 EST,5.09,39.0,5.13,39.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250203C00606000"""
2025-01-23 09:30:00.010557696 EST,0.5,15.0,0.53,28.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250203C00620000"""
2025-01-23 09:30:00.011454976 EST,0.7,21.0,0.77,21.0,null,null,null,null,2025-01-23 09:30:00.002141696 EST,605.8,100.0,0,2025-01-23 09:30:00.001962240 EST,605.76,600.0,605.8,1600.0,"""O:SPY250203P00587000"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-23 15:59:59.978066944 EST,16.25,33.0,16.4,33.0,null,null,null,null,2025-01-23 15:59:59.888601600 EST,609.71,100.0,0,2025-01-23 15:59:59.886036480 EST,609.71,1600.0,609.74,200.0,"""O:SPY250203C00595000"""
2025-01-23 15:59:59.978703360 EST,15.36,33.0,15.52,33.0,null,null,null,null,2025-01-23 15:59:59.978344960 EST,609.71,100.0,0,2025-01-23 15:59:59.978341120 EST,609.71,16300.0,609.73,300.0,"""O:SPY250203C00596000"""
2025-01-23 15:59:59.978835712 EST,3.29,50.0,3.31,62.0,null,null,null,null,2025-01-23 15:59:59.978344960 EST,609.71,100.0,0,2025-01-23 15:59:59.978341120 EST,609.71,16300.0,609.73,300.0,"""O:SPY250203P00607000"""


#### Expiry 2025-02-03 Ref Data

opt_ticker,nms5,is_american,is_call,strike,dte,prev_spot,ticker
str,f32,bool,bool,f32,i16,f64,str
"""O:SPY250203C00590000""",-1.282106,true,true,0.972891,7,606.440002,"""SPY"""
"""O:SPY250203C00591000""",-1.203104,true,true,0.97454,7,606.440002,"""SPY"""
"""O:SPY250203C00592000""",-1.124236,true,true,0.976189,7,606.440002,"""SPY"""
"""O:SPY250203C00595000""",-0.888429,true,true,0.981136,7,606.440002,"""SPY"""
"""O:SPY250203C00596000""",-0.810091,true,true,0.982785,7,606.440002,"""SPY"""
…,…,…,…,…,…,…,…
"""O:SPY250203P00607000""",0.043058,true,false,1.000923,7,606.440002,"""SPY"""
"""O:SPY250203P00608000""",0.119849,true,false,1.002572,7,606.440002,"""SPY"""
"""O:SPY250203P00609000""",0.196514,true,false,1.004221,7,606.440002,"""SPY"""


'\n(-1) add_business_days: 2025-01-23 -1->-1 True True [datetime.date(2025, 1, 23), datetime.date(2025, 1, 22), datetime.date(2025, 1, 21), datetime.date(2025, 1, 17), datetime.date(2025, 1, 16)] -> 2025-01-16\n(-2) add_business_days: 2025-01-23 -1->1 True True [datetime.date(2025, 1, 16), datetime.date(2025, 1, 17), datetime.date(2025, 1, 21), datetime.date(2025, 1, 22), datetime.date(2025, 1, 23)] -> 2025-01-17\n'

In [ ]:
"""
from quantdata.biz_calendar import DateRange

date_range = DateRange(date(2025,1,1), date(2025,12,31))
dates      = date_range.get_business_days()
dresults   = {}

for d in dates:
    dresults[d] = generate_day("SPY", d, KalmanConfig(), pool_config=pool_config, verbose=Context.all)
"""


In [ ]:
from quantdata.biz_calendar import add_business_days, prev_business_day, next_business_day

d = date(2025,1,23)
print(add_business_days(d,1,count_start=True))
print(add_business_days(d,1,count_start=False))
print(next_business_day(d,count_start=True))
print(next_business_day(d,count_start=False))

print(add_business_days(d,-1,count_start=True))
print(add_business_days(d,-1,count_start=False))
print(prev_business_day(d, count_start=True))
print(prev_business_day(d, count_start=False))


In [ ]:
from quantdata.biz_calendar import _BUSINESS_DAYS_CACHE

for k, c in _BUSINESS_DAYS_CACHE.items():
    print(k)
    print(c)